In [ ]:
import json
import logging
import sys
import os
import importlib
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
sys.path.append(os.path.abspath(".."))
import config
importlib.reload(config)

from config import (
    DATASET_CLEAN_DIR,
    FEATURE_COLUMNS,
    MODELS_DIR,
    TARGET_COLUMN
)
from training.temporal_cv import expanding_month_splits


In [ ]:
df = pd.read_csv(DATASET_CLEAN_DIR / "features_ispa_monthly.csv")
df["month_start"] = pd.to_datetime(df["month_start"])

# 1. Temporal Split (Train: < 2025-10-01, Test: >= 2025-10-01)
train_mask = df["month_start"] < "2025-10-01"
test_mask = df["month_start"] >= "2025-10-01"

train_df = df[train_mask].copy()
test_df = df[test_mask].copy()

X_train, y_train = train_df[FEATURE_COLUMNS], train_df[TARGET_COLUMN]
X_test, y_test = test_df[FEATURE_COLUMNS], test_df[TARGET_COLUMN]

print(f"Training samples: {len(train_df)} rows")
print(f"Testing samples : {len(test_df)} rows")
# Tuning memakai masa sebelum validasi blend; Okt–Des 2025 tetap holdout akhir.
months_train = sorted(train_df["month_start"].unique())
blend_start = months_train[-1]
dev_df = train_df[train_df["month_start"] < blend_start].copy()
blend_df = train_df[train_df["month_start"] >= blend_start].copy()
X_dev, y_dev = dev_df[FEATURE_COLUMNS], dev_df[TARGET_COLUMN]
X_blend, y_blend = blend_df[FEATURE_COLUMNS], blend_df[TARGET_COLUMN]
cv_splits = expanding_month_splits(dev_df, n_splits=3, min_train_months=2)


In [ ]:
# Benchmark Single Sub-Models for ISPA
m_rf = RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=1, random_state=42, n_jobs=-1).fit(X_train, y_train)
m_xgb = XGBRegressor(learning_rate=0.05, n_estimators=200, max_depth=4, min_child_weight=7, subsample=0.8, colsample_bytree=0.9, reg_alpha=10, reg_lambda=5, random_state=42).fit(X_train, y_train)
m_enet = make_pipeline(StandardScaler(), ElasticNet(alpha=0.2, l1_ratio=0.5, max_iter=10000, random_state=42)).fit(X_train, y_train)

p_rf = np.clip(m_rf.predict(X_test), 0, None)
p_xgb = np.clip(m_xgb.predict(X_test), 0, None)
p_enet = np.clip(m_enet.predict(X_test), 0, None)

print("=== BENCHMARK SINGLE SUB-MODELS FOR ISPA ===")
print(f"Random Forest    -> MAE: {mean_absolute_error(y_test, p_rf):.2f}, RMSE: {np.sqrt(mean_squared_error(y_test, p_rf)):.2f}, R2: {r2_score(y_test, p_rf):.4f}")
print(f"XGBoost          -> MAE: {mean_absolute_error(y_test, p_xgb):.2f}, RMSE: {np.sqrt(mean_squared_error(y_test, p_xgb)):.2f}, R2: {r2_score(y_test, p_xgb):.4f}")
print(f"ElasticNet       -> MAE: {mean_absolute_error(y_test, p_enet):.2f}, RMSE: {np.sqrt(mean_squared_error(y_test, p_enet)):.2f}, R2: {r2_score(y_test, p_enet):.4f}")

In [ ]:
# -------------------------------------------------------------
# HYPERPARAMETER TUNING (RandomizedSearchCV) & WEIGHT OPTIMIZATION FOR ALL ISPA SUB-MODELS
# -------------------------------------------------------------

# 1. Tune Random Forest Sub-Model
param_grid_rf = {'n_estimators': [100, 200, 300], 'max_depth': [6, 8, 10], 'min_samples_leaf': [1, 2, 3]}
rs_rf = RandomizedSearchCV(RandomForestRegressor(random_state=42), param_distributions=param_grid_rf, n_iter=8, cv=cv_splits, random_state=42, scoring='r2', n_jobs=-1)
rs_rf.fit(X_dev, y_dev)
print("=== 1. RANDOMIZED SEARCH CV FOR RANDOM FOREST SUB-MODEL ===")
print(f"Best Random Forest Params: {rs_rf.best_params_}")
print(f"Best Random Forest CV Score (R2): {rs_rf.best_score_:.4f}\n")

# 2. Tune XGBoost Sub-Model
param_grid_xgb = {'n_estimators': [100, 150, 200], 'max_depth': [3, 4, 5], 'learning_rate': [0.03, 0.05, 0.1], 'min_child_weight': [3, 5, 7]}
rs_xgb = RandomizedSearchCV(XGBRegressor(random_state=42), param_distributions=param_grid_xgb, n_iter=8, cv=cv_splits, random_state=42, scoring='r2', n_jobs=-1)
rs_xgb.fit(X_dev, y_dev)
print("=== 2. RANDOMIZED SEARCH CV FOR XGBOOST SUB-MODEL ===")
print(f"Best XGBoost Params: {rs_xgb.best_params_}")
print(f"Best XGBoost CV Score (R2): {rs_xgb.best_score_:.4f}\n")

# 3. Tune ElasticNet Sub-Model
param_grid_enet = {'elasticnet__alpha': [0.1, 0.2, 0.5, 1.0], 'elasticnet__l1_ratio': [0.1, 0.3, 0.5, 0.7]}
rs_enet = RandomizedSearchCV(make_pipeline(StandardScaler(), ElasticNet(max_iter=10000, random_state=42)), param_distributions=param_grid_enet, n_iter=6, cv=cv_splits, random_state=42, scoring='r2', n_jobs=-1)
rs_enet.fit(X_dev, y_dev)
print("=== 3. RANDOMIZED SEARCH CV FOR ELASTICNET SUB-MODEL ===")
print(f"Best ElasticNet Params: {rs_enet.best_params_}")
print(f"Best ElasticNet CV Score (R2): {rs_enet.best_score_:.4f}\n")

# 4. Fit Best Models & Optimize Ensemble Weights
m_rf = rs_rf.best_estimator_
m_xgb = rs_xgb.best_estimator_
m_enet = rs_enet.best_estimator_

p_rf = np.clip(m_rf.predict(X_blend), 0, None)
p_xgb = np.clip(m_xgb.predict(X_blend), 0, None)
p_enet = np.clip(m_enet.predict(X_blend), 0, None)

def loss_func(weights):
    w1, w2, w3 = weights
    pred = w1 * p_rf + w2 * p_xgb + w3 * p_enet
    pred = np.clip(pred, 0, None)
    return mean_squared_error(y_blend, pred)

constraints = ({'type': 'eq', 'fun': lambda w: 1.0 - sum(w)})
bounds = [(0.10, 0.70) for _ in range(3)]
res = minimize(loss_func, [0.50, 0.35, 0.15], method='SLSQP', bounds=bounds, constraints=constraints)
opt_w = res.x

print("=== 4. OPTIMIZING ENSEMBLE WEIGHTS (SciPy SLSQP) ===")
print(f"Optimal Random Forest Weight : {opt_w[0]:.4f}")
print(f"Optimal XGBoost Weight       : {opt_w[1]:.4f}")
print(f"Optimal ElasticNet Weight    : {opt_w[2]:.4f}")

# Setelah bobot terkunci, latih ulang pada seluruh periode latih.
m_rf.fit(X_train, y_train)
m_xgb.fit(X_train, y_train)
m_enet.fit(X_train, y_train)
p_rf = np.clip(m_rf.predict(X_test), 0, None)
p_xgb = np.clip(m_xgb.predict(X_test), 0, None)
p_enet = np.clip(m_enet.predict(X_test), 0, None)

p_opt = np.clip(opt_w[0] * p_rf + opt_w[1] * p_xgb + opt_w[2] * p_enet, 0, None)
print(f"\nEnsemble Blended Metrics:")
print(f"MAE : {mean_absolute_error(y_test, p_opt):.4f} cases/month")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, p_opt)):.4f}")
print(f"R2  : {r2_score(y_test, p_opt):.4f} ({r2_score(y_test, p_opt)*100:.2f}%)")

In [ ]:
from training.ensemble import ISPAEnsembleModel

model = ISPAEnsembleModel()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== EVALUATION OF ENSEMBLE MODEL FOR ISPA ===")
print(f"MAE : {mae:.4f} cases/month")
print(f"RMSE: {rmse:.4f}")
print(f"R2  : {r2:.4f} ({r2*100:.2f}%)")

importances = model.feature_importances_
feat_importance_df = (
    pd.DataFrame({"feature": FEATURE_COLUMNS, "importance": importances})
    .sort_values(by="importance", ascending=False)
    .reset_index(drop=True)
)
print("\nTop 5 Feature Importances:\n" + feat_importance_df.head(5).to_string(index=False))